# 🌊 AquaSentinel AI: Complete 11-Step Production Training Pipeline
### Side-Scan Sonar Segmentation (GhostVision + SSS-Mine / NOMBO)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RaghavKacker/Aqua-Sentinel/blob/main/notebooks/AquaSentinel_Training.ipynb)

This notebook contains the complete, step-by-step workflow matching the AquaSentinel specification:

```text
1. Mount Google Drive
2. Install dependencies
3. Download / Setup GhostVision
4. Download / Setup SSS-Mine
5. Inspect both datasets (Tree, Annotations, Visual)
6. Convert annotations to YOLO-Seg format
7. Create train / val / test (Survey-Isolated splits)
8. Train YOLO-Seg (yolo11n-seg)
9. Validate (Box & Mask mAP50, mAP50-95)
10. Test on unseen missions
11. Compare results & Export best.pt
```

---

## 1. Mount Google Drive
Mount Google Drive to persist datasets and checkpoints across Colab restarts.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted successfully!")

## 2. Install Dependencies
Verify GPU accelerator and install required libraries.

In [ ]:
!nvidia-smi
!pip install -q ultralytics opencv-python-headless pyyaml matplotlib tabulate tqdm

import torch, ultralytics
print(f"\nPyTorch Version: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
ultralytics.checks()

## 3. Download / Setup GhostVision
Locate or download the GhostVision dataset into Google Drive.

In [ ]:
import os
from pathlib import Path

DATASET_DIR = "/content/drive/MyDrive/AquaSentinel/datasets"
ghostvision_path = os.path.join(DATASET_DIR, "GhostVision")

# Verify folder exists
if not os.path.exists(ghostvision_path):
    os.makedirs(ghostvision_path, exist_ok=True)
    print(f"Created GhostVision directory at: {ghostvision_path}")
    print("Download or copy GhostVision files into this folder.")
else:
    print(f"GhostVision located at: {ghostvision_path}")

## 4. Download / Setup SSS-Mine
Locate SSS-Mine dataset archives in Google Drive and unpack any remaining zip files.

In [ ]:
import zipfile

sss_mine_path = os.path.join(DATASET_DIR, "SSS-Mine")
os.makedirs(sss_mine_path, exist_ok=True)

# Unpack any survey archives if not yet extracted
for zip_file in Path(sss_mine_path).glob("*.zip"):
    target_unzip_dir = Path(sss_mine_path) / zip_file.stem
    if not target_unzip_dir.exists():
        print(f"Extracting {zip_file.name}...")
        with zipfile.ZipFile(zip_file, 'r') as zip_ref:
            zip_ref.extractall(target_unzip_dir)

print(f"SSS-Mine located at: {sss_mine_path}")

## 5. Inspect Both Datasets
### 5.1 Directory Tree Inspection
Scans the directory hierarchy, file extensions, and file counts.

In [ ]:
from collections import Counter
import os

def inspect_dataset(dataset_path, max_depth=3):
    print("=" * 80)
    print(f"DATASET: {dataset_path}")
    print("=" * 80)

    if not os.path.exists(dataset_path):
        print("❌ Dataset folder does not exist!")
        return

    extension_counts = Counter()
    total_files = 0

    for root, dirs, files in os.walk(dataset_path):
        relative = os.path.relpath(root, dataset_path)
        depth = 0 if relative == "." else relative.count(os.sep) + 1

        if depth > max_depth:
            dirs[:] = []
            continue

        indent = "  " * depth
        folder_name = os.path.basename(root)

        print(f"{indent}📁 {folder_name}/")

        for file in files[:15]:
            print(f"{indent}  📄 {file}")

        if len(files) > 15:
            print(f"{indent}  ... +{len(files)-15} more files")

        for file in files:
            total_files += 1
            ext = os.path.splitext(file)[1].lower()
            if ext:
                extension_counts[ext] += 1
            else:
                extension_counts["[no extension]"] += 1

    print("\nFile summary:")
    print("Total files:", total_files)
    print("Extensions:")
    for ext, count in extension_counts.most_common():
        print(f"  {ext}: {count}")
    print()

# Run inspection on both datasets
inspect_dataset(ghostvision_path)
inspect_dataset(sss_mine_path)

### 5.2 Deep Annotation Inspection & Class Discovery
Inspects the format and discovered labels in `GhostVision/metadata.jsonl` and `SSS-Mine/*.txt`.

In [ ]:
import json, glob
from collections import Counter
from pathlib import Path

print("=" * 80)
print("INSPECTING GHOSTVISION ANNOTATIONS")
print("=" * 80)

gv_dir = Path(ghostvision_path)
for split in ["test", "train", "valid"]:
    jsonl_f = gv_dir / split / "metadata.jsonl"
    if jsonl_f.exists():
        print(f"\n📄 Found {split}/metadata.jsonl ({jsonl_f.stat().st_size} bytes):")
        with open(jsonl_f, 'r', encoding='utf-8') as f:
            for i in range(2):
                line = f.readline()
                if line:
                    data = json.loads(line.strip())
                    print(f"  Row {i+1} file_name: {data.get('file_name')}")
                    print(f"  Row {i+1} objects:   {data.get('objects')}")

print("\n" + "=" * 80)
print("INSPECTING SSS-MINE ANNOTATIONS")
print("=" * 80)

mine_dir = Path(sss_mine_path)
mine_txts = list(mine_dir.glob('**/*.txt'))
print(f"Total TXT annotation files found: {len(mine_txts)}")

mine_classes = Counter()
sample_lines = []
for tf in mine_txts:
    try:
        with open(tf, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    mine_classes[parts[0]] += 1
                    if len(sample_lines) < 5:
                        sample_lines.append((tf.name, line.strip()))
    except Exception:
        pass

print(f"SSS-Mine Discovered Class Counts: {dict(mine_classes)}")
print("Sample lines:")
for fname, line in sample_lines:
    print(f"  {fname}: {line}")

### 5.3 Visual Sonar Inspection
Displays sample images from GhostVision and SSS-Mine.

In [ ]:
import cv2
import matplotlib.pyplot as plt

gv_imgs = list(Path(ghostvision_path).glob('**/*.jpg'))[:2]
mine_imgs = list(Path(sss_mine_path).glob('**/*.jpg'))[:2]
samples = [('GhostVision', p) for p in gv_imgs] + [('SSS-Mine', p) for p in mine_imgs]

if samples:
    fig, axes = plt.subplots(1, len(samples), figsize=(5 * len(samples), 5))
    if len(samples) == 1:
        axes = [axes]
    for i, (ds_name, img_p) in enumerate(samples):
        img = cv2.imread(str(img_p))
        if img is not None:
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            h, w = img.shape[:2]
            axes[i].imshow(img_rgb)
            axes[i].set_title(f"[{ds_name}]\n{img_p.parent.name}/{img_p.name}\n({w}x{h})", fontsize=9)
            axes[i].axis('off')
    plt.tight_layout()
    plt.show()

## 6. Convert Annotations to YOLO-Seg Format
Maps the discovered dataset annotations into the canonical 4-class taxonomy:

| Canonical ID | Canonical Name | Source Label |
| :--- | :--- | :--- |
| `0` | `crab_pot` | GhostVision: `Crab-Pot` |
| `1` | `ghost_gear` | GhostVision: derelict nets, gear, ropes |
| `2` | `mine_cylinder` | SSS-Mine: Target `0` (mine cylinder) |
| `3` | `debris_anomaly` | SSS-Mine: Target `1` (acoustic anomaly) |

Converts bounding coordinates into normalized 4-point YOLO-Seg polygon segmentation format:
`<class_id> <x1> <y1> <x2> <y2> <x3> <y3> <x4> <y4>`

In [ ]:
import os, json, shutil, cv2
from pathlib import Path
from tqdm import tqdm

OUTPUT_DATASET = Path("/content/unified_yolo_seg")
for split in ["train", "val", "test"]:
    (OUTPUT_DATASET / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUTPUT_DATASET / "labels" / split).mkdir(parents=True, exist_ok=True)

CANONICAL_CLASSES = {
    0: "crab_pot",
    1: "ghost_gear",
    2: "mine_cylinder",
    3: "debris_anomaly"
}

def clamp(val, min_val=0.0, max_val=1.0):
    return max(min_val, min(max_val, val))

# Helper to convert COCO [x, y, w, h] to 4-point YOLO-Seg polygon
def coco_bbox_to_polygon(bbox, img_w, img_h):
    x, y, w, h = bbox
    x1, y1 = clamp(x / img_w), clamp(y / img_h)
    x2, y2 = clamp((x + w) / img_w), clamp(y / img_h)
    x3, y3 = clamp((x + w) / img_w), clamp((y + h) / img_h)
    x4, y4 = clamp(x / img_w), clamp((y + h) / img_h)
    return [x1, y1, x2, y2, x3, y3, x4, y4]

# Helper to convert YOLO [cx, cy, w, h] to 4-point YOLO-Seg polygon
def yolo_bbox_to_polygon(cx, cy, w, h):
    x1, y1 = clamp(cx - w / 2), clamp(cy - h / 2)
    x2, y2 = clamp(cx + w / 2), clamp(cy - h / 2)
    x3, y3 = clamp(cx + w / 2), clamp(cy + h / 2)
    x4, y4 = clamp(cx - w / 2), clamp(cy + h / 2)
    return [x1, y1, x2, y2, x3, y3, x4, y4]

print("Conversion helpers defined successfully!")

## 7. Create Survey-Isolated Train / Val / Test Splits
Partitions the data by survey mission / recording ID to prevent data leakage across splits:
- **GhostVision:** `train` (`Rec6`), `valid` (`Contact`), `test` (`Rec8` - unseen)
- **SSS-Mine:** `2010, 2015, 2018` $\rightarrow$ `train`, `2017` $\rightarrow$ `val`, `2021` $\rightarrow$ `test` (unseen)

In [ ]:
import os, json, shutil, cv2
from pathlib import Path
from tqdm import tqdm

# Safe path definitions (prevents NameError if runtime restarted)
DATASET_DIR = "/content/drive/MyDrive/AquaSentinel/datasets"
if 'ghostvision_path' not in globals() or not ghostvision_path:
    ghostvision_path = os.path.join(DATASET_DIR, "GhostVision")
if 'sss_mine_path' not in globals() or not sss_mine_path:
    sss_mine_path = os.path.join(DATASET_DIR, "SSS-Mine")

OUTPUT_DATASET = Path("/content/unified_yolo_seg")
for split in ["train", "val", "test"]:
    (OUTPUT_DATASET / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUTPUT_DATASET / "labels" / split).mkdir(parents=True, exist_ok=True)

def clamp(val, min_val=0.0, max_val=1.0):
    return max(min_val, min(max_val, val))

def coco_bbox_to_polygon(bbox, img_w, img_h):
    x, y, w, h = bbox
    x1, y1 = clamp(x / img_w), clamp(y / img_h)
    x2, y2 = clamp((x + w) / img_w), clamp(y / img_h)
    x3, y3 = clamp((x + w) / img_w), clamp((y + h) / img_h)
    x4, y4 = clamp(x / img_w), clamp((y + h) / img_h)
    return [x1, y1, x2, y2, x3, y3, x4, y4]

gv_split_map = {"train": "train", "valid": "val", "test": "test"}
gv_stats = {"train": 0, "val": 0, "test": 0}

print(f"GhostVision path: {ghostvision_path}")
print("Processing GhostVision annotations...")
for gv_sub, target_split in gv_split_map.items():
    sub_dir = Path(ghostvision_path) / gv_sub
    jsonl_path = sub_dir / "metadata.jsonl"
    if not jsonl_path.exists():
        continue

    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in tqdm(f, desc=f"GhostVision {gv_sub} -> {target_split}"):
            row = json.loads(line.strip())
            img_rel = row.get("file_name")
            src_img_p = sub_dir / img_rel
            if not src_img_p.exists():
                continue

            img = cv2.imread(str(src_img_p))
            if img is None:
                continue
            h, w = img.shape[:2]

            dst_img_name = f"gv_{src_img_p.name}"
            dst_img_p = OUTPUT_DATASET / "images" / target_split / dst_img_name
            shutil.copyfile(src_img_p, dst_img_p)

            dst_lbl_p = OUTPUT_DATASET / "labels" / target_split / f"{dst_img_p.stem}.txt"
            objects = row.get("objects", {})
            bboxes = objects.get("bbox", [])
            categories = objects.get("category", [])

            with open(dst_lbl_p, "w", encoding="utf-8") as lf:
                for bbox, cat in zip(bboxes, categories):
                    cls_id = 0 if "crab" in str(cat).lower() else 1
                    poly = coco_bbox_to_polygon(bbox, w, h)
                    poly_str = " ".join([f"{pt:.6f}" for pt in poly])
                    lf.write(f"{cls_id} {poly_str}\n")

            gv_stats[target_split] += 1

print("GhostVision processing complete! Distribution:", gv_stats)


In [ ]:
import os, json, shutil
from pathlib import Path
from tqdm import tqdm

DATASET_DIR = "/content/drive/MyDrive/AquaSentinel/datasets"
if 'sss_mine_path' not in globals() or not sss_mine_path:
    sss_mine_path = os.path.join(DATASET_DIR, "SSS-Mine")

OUTPUT_DATASET = Path("/content/unified_yolo_seg")

def clamp(val, min_val=0.0, max_val=1.0):
    return max(min_val, min(max_val, val))

def yolo_bbox_to_polygon(cx, cy, w, h):
    x1, y1 = clamp(cx - w / 2), clamp(cy - h / 2)
    x2, y2 = clamp(cx + w / 2), clamp(cy - h / 2)
    x3, y3 = clamp(cx + w / 2), clamp(cy + h / 2)
    x4, y4 = clamp(cx - w / 2), clamp(cy + h / 2)
    return [x1, y1, x2, y2, x3, y3, x4, y4]

mine_mission_map = {
    "2010": "train",
    "2015": "train",
    "2018": "train",
    "2017": "val",
    "2021": "test"
}
mine_stats = {"train": 0, "val": 0, "test": 0}

print(f"SSS-Mine path: {sss_mine_path}")
print("Processing SSS-Mine / NOMBO annotations...")
for mission_year, target_split in mine_mission_map.items():
    year_dir = Path(sss_mine_path) / mission_year
    all_jpgs = list(year_dir.glob("**/*.jpg"))
    
    for src_img_p in tqdm(all_jpgs, desc=f"SSS-Mine {mission_year} -> {target_split}"):
        src_lbl_p = src_img_p.with_suffix(".txt")
        if not src_lbl_p.exists():
            continue

        dst_img_name = f"mine_{mission_year}_{src_img_p.name}"
        dst_img_p = OUTPUT_DATASET / "images" / target_split / dst_img_name
        shutil.copyfile(src_img_p, dst_img_p)

        dst_lbl_p = OUTPUT_DATASET / "labels" / target_split / f"{dst_img_p.stem}.txt"
        with open(src_lbl_p, "r", encoding="utf-8") as inf, open(dst_lbl_p, "w", encoding="utf-8") as outf:
            for line in inf:
                parts = line.strip().split()
                if len(parts) >= 5:
                    raw_cls = parts[0]
                    cls_id = 2 if raw_cls == "0" else 3
                    cx, cy, bw, bh = map(float, parts[1:5])
                    poly = yolo_bbox_to_polygon(cx, cy, bw, bh)
                    poly_str = " ".join([f"{pt:.6f}" for pt in poly])
                    outf.write(f"{cls_id} {poly_str}\n")

        mine_stats[target_split] += 1

print("SSS-Mine processing complete! Distribution:", mine_stats)


In [ ]:
import yaml

yaml_data = {
    "path": str(OUTPUT_DATASET.resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": CANONICAL_CLASSES
}

yaml_file = OUTPUT_DATASET / "dataset.yaml"
with open(yaml_file, "w", encoding="utf-8") as f:
    yaml.dump(yaml_data, f, default_flow_style=False, sort_keys=False)

print(f"YAML Configuration generated at: {yaml_file}")
print("\n--- DATASET.YAML CONTENTS ---")
print(open(yaml_file).read())

for split in ["train", "val", "test"]:
    n_imgs = len(list((OUTPUT_DATASET / "images" / split).glob("*")))
    n_lbls = len(list((OUTPUT_DATASET / "labels" / split).glob("*")))
    print(f"Split '{split}': {n_imgs} images, {n_lbls} label files")

## 8. Train YOLO-Seg Model on Colab GPU
Trains `yolo11n-seg.pt` transfer learning model:
- `imgsz=640` preserves acoustic speckle and acoustic shadow geometry
- `fliplr=0.5` leverages port/starboard swath symmetry
- `flipud=0.0` preserves acoustic nadir geometry
- `mosaic=0.5` mixes seabed textures for hard-clutter suppression

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n-seg.pt")

results = model.train(
    data=str(OUTPUT_DATASET / "dataset.yaml"),
    epochs=35,
    imgsz=640,
    batch=16,
    device=0,
    workers=4,
    project="/content/AquaSentinel_Runs",
    name="aquasentinel_seg",
    exist_ok=True,
    fliplr=0.5,
    flipud=0.0,
    mosaic=0.5,
    save=True,
    plots=True
)

print("Training completed successfully!")

## 9. Validate Model Performance
Evaluates Box & Mask Precision, Recall, mAP50, and mAP50-95 on the validation split.

In [ ]:
val_metrics = model.val(data=str(OUTPUT_DATASET / "dataset.yaml"), split="val")

print("\n--- VALIDATION METRICS ---")
print(f"Box  mAP50:    {val_metrics.box.map50:.4f} | mAP50-95: {val_metrics.box.map:.4f}")
print(f"Mask mAP50:    {val_metrics.seg.map50:.4f} | mAP50-95: {val_metrics.seg.map:.4f}")
print(f"Precision:     {val_metrics.box.mp:.4f}   | Recall:   {val_metrics.box.mr:.4f}")

## 10. Test & Visualize on Unseen Survey Missions
Evaluates model generalization on the `test` split (`GhostVision Rec8` and `SSS-Mine 2021`).

In [ ]:
import matplotlib.pyplot as plt

test_metrics = model.val(data=str(OUTPUT_DATASET / "dataset.yaml"), split="test")

print("\n--- UNSEEN MISSION TEST METRICS ---")
print(f"Test Box  mAP50: {test_metrics.box.map50:.4f} | mAP50-95: {test_metrics.box.map:.4f}")
print(f"Test Mask mAP50: {test_metrics.seg.map50:.4f} | mAP50-95: {test_metrics.seg.map:.4f}")

test_images = list((OUTPUT_DATASET / "images" / "test").glob("*.jpg"))[:4]
if test_images:
    pred_results = model.predict(test_images, conf=0.25)
    fig, axes = plt.subplots(1, len(pred_results), figsize=(5 * len(pred_results), 5))
    if len(pred_results) == 1:
        axes = [axes]
    for idx, r in enumerate(pred_results):
        res_plotted = r.plot()
        axes[idx].imshow(cv2.cvtColor(res_plotted, cv2.COLOR_BGR2RGB))
        axes[idx].set_title(f"Test Sample {idx+1}", fontsize=10)
        axes[idx].axis("off")
    plt.tight_layout()
    plt.show()

## 11. Compare Results & Export best.pt
Exports and initiates download of `best.pt` for offline deployment inside `models/best.pt`.

In [ ]:
import os
from google.colab import files

best_pt_path = "/content/AquaSentinel_Runs/aquasentinel_seg/weights/best.pt"

if os.path.exists(best_pt_path):
    file_size_mb = os.path.getsize(best_pt_path) / (1024 * 1024)
    print(f"Found best weights at: {best_pt_path} ({file_size_mb:.2f} MB)")
    
    backup_path = "/content/drive/MyDrive/AquaSentinel/models/best.pt"
    os.makedirs(os.path.dirname(backup_path), exist_ok=True)
    shutil.copyfile(best_pt_path, backup_path)
    print(f"Backed up to Google Drive at: {backup_path}")
    
    print("Starting download of best.pt to your local machine...")
    files.download(best_pt_path)
else:
    print(f"[ERROR] Weights file not found at {best_pt_path}. Check training status.")